In [ ]:
# Install the required library first
!pip install -q chembl_webresource_client pandas

import pandas as pd
from chembl_webresource_client.new_client import new_client

def fetch_chembl_data():
    print("Step 1: Searching for the Target Protein...")

    # 1. Initialize the target client
    target_api = new_client.target

    # Search for EGFR (Epidermal growth factor receptor)
    target_query = target_api.search('EGFR')
    targets = pd.DataFrame.from_dict(target_query)

    # 2. Select the specific Target ChEMBL ID
    # We want the SINGLE PROTEIN target for Human EGFR
    selected_target = targets[
        (targets['target_type'] == 'SINGLE PROTEIN') &
        (targets['organism'] == 'Homo sapiens')
    ].iloc[0]

    target_chembl_id = selected_target['target_chembl_id']
    print(f"Target Found: {selected_target['pref_name']}")
    print(f"Target ChEMBL ID: {target_chembl_id}\n")

    print("Step 2: Fetching Bioactivity Data (IC50)...")
    print("Note: This may take 2-3 minutes depending on API load.")

    # 3. Initialize the activity client
    activity_api = new_client.activity

    # Filter for our specific target and specifically for IC50 standard types
    res = activity_api.filter(
        target_chembl_id=target_chembl_id,
        standard_type="IC50",
        relation="="  # We only want exact measurements, not ">" or "<"
    )

    # Convert the API results directly into a Pandas DataFrame
    df = pd.DataFrame.from_dict(res)
    print(f"Raw data retrieved: {len(df)} records.\n")

    print("Step 3: Cleaning and Filtering Data...")

    # 4. Data Cleaning
    # Drop rows where there is no standard value (the actual IC50 number) or no SMILES string (the chemical structure)
    df_clean = df.dropna(subset=['standard_value', 'canonical_smiles']).copy()

    # Drop duplicate molecules (keep the first tested instance)
    df_clean = df_clean.drop_duplicates(subset=['canonical_smiles'])

    # Keep only the columns we actually need for machine learning
    columns_to_keep = [
        'molecule_chembl_id',
        'canonical_smiles',
        'standard_type',
        'standard_value',
        'standard_units'
    ]
    df_clean = df_clean[columns_to_keep]

    # Convert standard_value to numeric (the API returns it as a string)
    df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'])

    print(f"Clean data ready for Machine Learning: {len(df_clean)} unique molecules.")

    # 5. Save the data to a CSV for the next step of the project
    output_filename = f"{target_chembl_id}_IC50_bioactivity_data.csv"
    df_clean.to_csv(output_filename, index=False)
    print(f"\nData successfully saved to: {output_filename}")

    return df_clean

# Execute the pipeline
if __name__ == "__main__":
    clean_data = fetch_chembl_data()
    # Display the first 3 rows to verify
    print("\nData Preview:")
    print(clean_data.head(3))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Save directly to your Drive:
# df_clean.to_csv('/content/drive/MyDrive/chembl_data.csv', index=False)

In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Feature Engineering & Preprocessing with RDKit
# ---------------------------------------------------------------------------

# 1. Install RDKit and dependencies
!pip install -q rdkit pandas numpy matplotlib

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
import os

# 2. Load the dataset saved from Step 1
# Replace filename if your ChEMBL ID differed
input_file = 'CHEMBL203_IC50_bioactivity_data.csv'

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Cannot find {input_file}. Ensure Step 1 script ran successfully!")

df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows from {input_file}.")


# 3. Function to convert IC50 (nM) to pIC50 (-log10(M))
def convert_to_pic50(input_df):
    pIC50 = []

    for val in input_df['standard_value']:
        # Cap high values to 100,000,000 nM to prevent negative pIC50 values from noise
        if val > 100000000:
            val = 100000000
        # Prevent log(0) errors
        if val <= 0:
            val = 1e-9

        molar = val * 1e-9  # Convert nM to Molar
        pIC50.append(-np.log10(molar))

    input_df['pIC50'] = pIC50
    return input_df

print("\nConverting IC50 to pIC50...")
df = convert_to_pic50(df)


# 4. Function to calculate Lipinski Descriptors (Molecular Weight, LogP, NumHDonors, NumHAcceptors)
def calculate_lipinski(smiles_list):
    mol_data = []

    for sm in smiles_list:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hdonors = Descriptors.NumHDonors(mol)
            hacceptors = Descriptors.NumHAcceptors(mol)
            mol_data.append([mw, logp, hdonors, hacceptors])
        else:
            mol_data.append([np.nan, np.nan, np.nan, np.nan])

    descriptors_df = pd.DataFrame(
        mol_data,
        columns=['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
    )
    return descriptors_df

print("Calculating Lipinski Descriptors...")
lipinski_df = calculate_lipinski(df['canonical_smiles'])
df = pd.concat([df, lipinski_df], axis=1)


# 5. Function to generate Morgan Fingerprints (ECFP4 - Radius 2, 1024 bits)
def generate_morgan_fingerprints(smiles_list, radius=2, n_bits=1024):
    fingerprints = []

    for sm in smiles_list:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
            # Generate bit vector for Morgan Fingerprint (ECFP4 equivalent)
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            # Convert bit vector to numpy array
            arr = np.zeros((0,), dtype=int)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            fingerprints.append(arr)
        else:
            fingerprints.append(np.zeros((n_bits,), dtype=int))

    # Create DataFrame where each column represents 1 bit feature (Bit_0 to Bit_1023)
    bit_columns = [f"Bit_{i}" for i in range(n_bits)]
    fp_df = pd.DataFrame(fingerprints, columns=bit_columns)
    return fp_df

print("Generating 1024-bit Morgan Fingerprints (ECFP4)...")
fp_df = generate_morgan_fingerprints(df['canonical_smiles'])


# 6. Combine all features into a final dataset ready for Machine Learning
final_df = pd.concat([df, fp_df], axis=1)

# Drop any rows where SMILES parsing failed
final_df = final_df.dropna(subset=['MW', 'pIC50'])

# Save processed dataset
output_file = 'processed_qsar_dataset.csv'
final_df.to_csv(output_file, index=False)

print(f"\n--- SUCCESS ---")
print(f"Final Dataset Shape: {final_df.shape} (Rows, Columns)")
print(f"Features Generated: 4 Lipinski Descriptors + 1024 Fingerprint Bits")
print(f"Target Variable: pIC50")
print(f"Dataset saved to: {output_file}")

# Preview target distribution
print("\npIC50 Summary Stats:")
print(final_df['pIC50'].describe())

In [ ]:
# ---------------------------------------------------------------------------
# STEP 3: Machine Learning Model Training & Evaluation
# ---------------------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import os

# 1. Load the processed dataset
input_file = 'processed_qsar_dataset.csv'

if not os.path.exists(input_file):
    raise FileNotFoundError("Dataset missing. Ensure Step 2 ran successfully!")

df = pd.read_csv(input_file)

# 2. Define Features (X) and Target (y)
# Drop the target variable and any text/metadata columns to isolate the numeric features
X = df.drop(columns=['molecule_chembl_id', 'canonical_smiles',
                     'standard_type', 'standard_value',
                     'standard_units', 'pIC50'])
y = df['pIC50']

print(f"Input Features Shape (X): {X.shape}")
print(f"Target Variable Shape (y): {y.shape}")

# 3. Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining set size: {X_train.shape[0]} molecules")
print(f"Testing set size: {X_test.shape[0]} molecules")

# 4. Initialize and Train the Random Forest Model
print("\nTraining Random Forest Regressor... (This may take a minute)")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Make Predictions and Evaluate Model Performance
y_pred = model.predict(X_test)

# Calculate R-squared and Root Mean Square Error (RMSE)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- MODEL PERFORMANCE ---")
print(f"R-squared (R²): {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

# Optional: A simple scatter plot of Experimental vs Predicted values
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
plt.title("Experimental vs Predicted pIC50")
plt.xlabel("Experimental pIC50")
plt.ylabel("Predicted pIC50")
plt.show()

# 6. Feature Importance Visualization
# Which molecular features drove the model's predictions?
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

# Get the top 20 most important features
top_n = 20
top_indices = indices[:top_n]
top_features = [feature_names[i] for i in top_indices]
top_importances = importances[top_indices]

# Plot the Feature Importances
plt.figure(figsize=(10, 6))
plt.title(f"Top {top_n} Most Important Molecular Features")
plt.bar(range(top_n), top_importances, align="center")
plt.xticks(range(top_n), top_features, rotation=45, ha='right')
plt.xlabel("Feature (Lipinski Descriptor or Morgan Bit)")
plt.ylabel("Relative Importance")
plt.tight_layout()
plt.show()